In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# 1. SETTINGS
# ============================================================

NUM_REVIEWS = 100_000

VOCAB_SIZE = 20_000
MAX_LENGTH = 150

EMBEDDING_DIM = 8
LSTM_UNITS = 16
DENSE_UNITS = 16

BATCH_SIZE = 64
EPOCHS = 10

RANDOM_STATE = 42


# ============================================================
# 2. LOAD AMAZON POLARITY DATASET
# ============================================================

print("Loading Amazon Polarity dataset...")

dataset = load_dataset("mteb/amazon_polarity")

# The original dataset has 3.6 million training examples.
# We only use 100,000.
data = dataset["train"].shuffle(seed=RANDOM_STATE).select(
    range(NUM_REVIEWS)
)

print(f"Number of reviews: {len(data)}")


# ============================================================
# 3. CONVERT TO PANDAS
# ============================================================

df = pd.DataFrame(data)

print("\nDataset columns:")
print(df.columns)

print("\nFirst 5 reviews:")
print(df.head())


# ============================================================
# 4. PREPARE TEXT AND LABELS
# ============================================================

texts = df["text"].astype(str).values
labels = df["label"].astype(int).values

print("\nLabel distribution:")
print(pd.Series(labels).value_counts())


# ============================================================
# 5. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    texts,
    labels,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("\nDataset split:")
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))


# ============================================================
# 6. TOKENIZER
# ============================================================

print("\nCreating tokenizer...")

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)


# ============================================================
# 7. TEXT -> INTEGER SEQUENCES
# ============================================================

print("Converting reviews to sequences...")

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)


# ============================================================
# 8. PADDING
# ============================================================

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print("\nInput shape:")
print("Training:", X_train_pad.shape)
print("Validation:", X_val_pad.shape)
print("Testing:", X_test_pad.shape)


# ============================================================
# 9. BUILD THE NEURAL NETWORK
# ============================================================

print("\nBuilding model...")

model = Sequential([

    # Token IDs -> 8-dimensional vectors
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_LENGTH
    ),

    # LSTM with 16 units
    LSTM(LSTM_UNITS),

    # Dense layer with ReLU
    Dense(
        DENSE_UNITS,
        activation="relu"
    ),

    # Binary classification
    Dense(
        1,
        activation="sigmoid"
    )
])


# ============================================================
# 10. COMPILE
# ============================================================

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# ============================================================
# 11. DISPLAY MODEL
# ============================================================

model.summary()


# ============================================================
# 12. EARLY STOPPING
# ============================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)


# ============================================================
# 13. TRAIN
# ============================================================

print("\nTraining model...")

history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1
)


# ============================================================
# 14. TEST SET EVALUATION
# ============================================================

print("\nEvaluating model...")

test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test,
    verbose=0
)

print("\nTest Loss:", test_loss)
print("Test Accuracy:", test_accuracy)


# ============================================================
# 15. PREDICTIONS
# ============================================================

probabilities = model.predict(
    X_test_pad,
    verbose=0
)

predictions = (probabilities >= 0.5).astype(int).flatten()


# ============================================================
# 16. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions,
        target_names=[
            "Negative",
            "Positive"
        ]
    )
)


# ============================================================
# 17. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    predictions
)

print("\nConfusion Matrix:")
print(cm)


# ============================================================
# 18. SAVE KERAS MODEL
# ============================================================

os.makedirs("saved_model", exist_ok=True)

model.save(
    "saved_model/amazon_lstm.keras"
)

print("\nModel saved:")
print("saved_model/amazon_lstm.keras")


# ============================================================
# 19. SAVE TOKENIZER
# ============================================================

with open(
    "saved_model/tokenizer.pkl",
    "wb"
) as file:

    pickle.dump(
        tokenizer,
        file
    )

print("Tokenizer saved:")
print("saved_model/tokenizer.pkl")


# ============================================================
# 20. SAVE MODEL INFORMATION
# ============================================================

model_info = {
    "vocab_size": VOCAB_SIZE,
    "max_length": MAX_LENGTH,
    "embedding_dim": EMBEDDING_DIM,
    "lstm_units": LSTM_UNITS,
    "dense_units": DENSE_UNITS,
    "num_reviews": NUM_REVIEWS,
    "test_accuracy": float(test_accuracy)
}

with open(
    "saved_model/model_info.pkl",
    "wb"
) as file:

    pickle.dump(
        model_info,
        file
    )

print("Model information saved:")
print("saved_model/model_info.pkl")

print("\nTraining complete!")

Loading Amazon Polarity dataset...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/6.76k [00:00<?, ?B/s]

data/train-00000-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  255MB            

data/train-00000-of-00004.parquet: downloading bytes:           |  0.00B            

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training model...
Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 35s 27ms/step - accuracy: 0.5026 - loss: 0.6931 - val_accuracy: 0.5003 - val_loss: 0.6920
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 44s 29ms/step - accuracy: 0.5677 - loss: 0.6575 - val_accuracy: 0.6090 - val_loss: 0.6413
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 34s 27ms/step - accuracy: 0.5429 - loss: 0.6714 - val_accuracy: 0.5170 - val_loss: 0.6921
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 41s 27ms/step - accuracy: 0.5880 - loss: 0.6459 - val_accuracy: 0.8232 - val_loss: 0.4476
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 40s 26ms/step - accuracy: 0.8781 - loss: 0.3018 - val_accuracy: 0.8920 - val_loss: 0.2658
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 32s 26ms/step - accuracy: 0.9234 - loss: 0.2046 - val_accuracy: 0.9035 - val_loss: 0.2470
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 32s 26ms/step - accuracy: 0.9379 - loss: 0.1699 - val_accuracy: 0.9015 - val_loss: 0.2591
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 35s 28ms/step -

In [ ]:
import pickle
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.sequence import pad_sequences


# ============================================================
# LOAD MODEL
# ============================================================

model = tf.keras.models.load_model(
    "saved_model/amazon_lstm.keras"
)


# ============================================================
# LOAD TOKENIZER
# ============================================================

with open(
    "saved_model/tokenizer.pkl",
    "rb"
) as file:

    tokenizer = pickle.load(file)


# ============================================================
# SETTINGS
# ============================================================

MAX_LENGTH = 150


# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict_sentiment(review):

    # Convert text to sequence
    sequence = tokenizer.texts_to_sequences(
        [review]
    )

    # Pad sequence
    padded = pad_sequences(
        sequence,
        maxlen=MAX_LENGTH,
        padding="post",
        truncating="post"
    )

    # Get probability
    probability = model.predict(
        padded,
        verbose=0
    )[0][0]

    # Classification
    if probability >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    return sentiment, float(probability)


# ============================================================
# TEST REVIEWS
# ============================================================

reviews = [
    "This product is absolutely fantastic. I love it!",

    "Terrible product. It stopped working after two days.",

    "The quality is excellent and I would definitely buy it again.",

    "Waste of money. Very disappointed."
]


for review in reviews:

    sentiment, probability = predict_sentiment(
        review
    )

    print("\nReview:")
    print(review)

    print("Sentiment:", sentiment)
    print("Probability:", probability)


Review:
This product is absolutely fantastic. I love it!
Sentiment: Positive
Probability: 0.9825038313865662

Review:
Terrible product. It stopped working after two days.
Sentiment: Negative
Probability: 0.012836637906730175

Review:
The quality is excellent and I would definitely buy it again.
Sentiment: Positive
Probability: 0.9866839051246643

Review:
Waste of money. Very disappointed.
Sentiment: Negative
Probability: 0.008160018362104893


# Model Architecture and Design Choices

## 1. Problem Definition

This project performs **binary sentiment classification** of Amazon product reviews.  
Each review is classified as either:

- **0 → Negative**
- **1 → Positive**

The model is trained using 100,000 Amazon Polarity reviews.

---

## 2. Model Architecture

The proposed neural network follows this architecture:

**Review Text → Tokenizer → Padding → Embedding(8) → LSTM(16) → Dense(16, ReLU) → Dense(1, Sigmoid)**

### Layer-wise Description

| Component | Description |
|---|---|
| **Tokenizer** | Converts words in the review into integer IDs. |
| **Padding** | Makes all review sequences the same length (150 tokens). |
| **Embedding(8)** | Converts each token into an 8-dimensional learned vector representation. |
| **LSTM(16)** | Processes the sequence and learns relationships between words while retaining relevant context. |
| **Dense(16, ReLU)** | Learns nonlinear combinations of the features extracted by the LSTM. |
| **Dense(1, Sigmoid)** | Produces a value between 0 and 1 for binary sentiment classification. |

---

## 3. Why These Components Were Chosen

### Tokenizer
Neural networks require numerical input, so the tokenizer converts textual reviews into numerical sequences. An **OOV (Out-of-Vocabulary)** token is used to handle unseen words.

### Embedding Dimension = 8
An embedding layer provides a dense representation of words instead of treating word IDs as meaningful numerical values. A dimension of **8** was selected to keep the model small and computationally efficient.

### LSTM = 16 Units
LSTM (**Long Short-Term Memory**) is a type of recurrent neural network designed for sequential data. It was selected because word order and context are important in sentiment analysis. **16 units** provide a lightweight architecture suitable for this project.

### Dense Layer = 16 Neurons
The dense layer further processes the representation generated by the LSTM. **ReLU (Rectified Linear Unit)** is used to introduce non-linearity while remaining computationally efficient.

### Sigmoid Output
Since this is a binary classification problem, a single output neuron with **sigmoid activation** is appropriate. Its output ranges from 0 to 1 and is interpreted as the estimated probability of the positive class.

---

## 4. Training Configuration

- **Optimizer:** Adam
- **Loss Function:** Binary Cross-Entropy
- **Batch Size:** 64
- **Maximum Epochs:** 10
- **Early Stopping:** Used to reduce overfitting
- **Vocabulary Size:** 20,000
- **Maximum Sequence Length:** 150

### Why Binary Cross-Entropy?

Binary cross-entropy is suitable for problems where the target has two classes. It measures the difference between the actual label and the probability predicted by the sigmoid output.

### Why Adam?

Adam is an adaptive optimization algorithm that generally provides fast and stable convergence with minimal manual tuning.

---

## 5. Advantages of the Architecture

- Lightweight and relatively fast to train.
- LSTM can capture sequential relationships in review text.
- Embeddings provide learned word representations.
- Suitable for binary sentiment classification.
- Easy to save and reuse for predictions.
- Can be extended with larger embeddings, more LSTM units, dropout, or more advanced architectures.

---

## 6. Overall Data Flow

**Raw Review → Tokenization → Integer Sequence → Padding → Word Embeddings → LSTM Feature Representation → Dense Layer → Sigmoid → Positive/Negative**